<a href="https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabtahilrehman/Nabtahil-flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.85 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [ ]:
%cd flyrank-ml-internship-starter

/content/flyrank-ml-internship-starter


In [ ]:
!find . -name "content_refresh_anonymized.csv"

./data/raw/content_refresh_anonymized.csv


In [ ]:
## Distributions

'''I examined the distributions of search volume, average position, and CTR because these are important signals for identifying SEO opportunities.

The distributions appear right-skewed (heavy-tailed), meaning a small number of pages receive very large values while most pages have relatively low values.

This suggests that averages alone may not fully represent the data, so percentile and bucket-based analysis are useful for signal testing.'''

'I examined the distributions of search volume, average position, and CTR because these are important signals for identifying SEO opportunities.\n\nThe distributions appear right-skewed (heavy-tailed), meaning a small number of pages receive very large values while most pages have relatively low values.\n\nThis suggests that averages alone may not fully represent the data, so percentile and bucket-based analysis are useful for signal testing.'

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
## Signal Test 1: Search Volume

'''Hypothesis:
Pages with higher search volume provide greater SEO opportunity.

Verdict: CONFIRMED

Observed results suggest that higher search-volume pages generally receive greater visibility and traffic potential.

---

## Signal Test 2: Average Position

Hypothesis:
Pages closer to page one generate stronger click performance.

Verdict: CONFIRMED

Observed CTR tends to improve as ranking position improves.

---

## Signal Test 3: CTR

Hypothesis:
Pages with lower CTR may provide optimisation opportunities.

Verdict: MIXED

Some low-CTR pages appear to have opportunity, but CTR may also be affected by SERP features and search intent.'''

'Hypothesis:\nPages with higher search volume provide greater SEO opportunity.\n\nVerdict: CONFIRMED\n\nObserved results suggest that higher search-volume pages generally receive greater visibility and traffic potential.\n\n---\n\n## Signal Test 2: Average Position\n\nHypothesis:\nPages closer to page one generate stronger click performance.\n\nVerdict: CONFIRMED\n\nObserved CTR tends to improve as ranking position improves.\n\n---\n\n## Signal Test 3: CTR\n\nHypothesis:\nPages with lower CTR may provide optimisation opportunities.\n\nVerdict: MIXED\n\nSome low-CTR pages appear to have opportunity, but CTR may also be affected by SERP features and search intent.'

In [ ]:
# Signal 1: Search Volume

volume_bucket = pd.qcut(
    df["search_volume"].fillna(0),
    q=4,
    duplicates="drop"
)

volume_test = (
    df.groupby(volume_bucket)
      .agg(
          n=("content_id", "count"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean")
      )
)

display(volume_test)


# Signal 2: Average Position

position_bucket = pd.qcut(
    df["avg_position"],
    q=4,
    duplicates="drop"
)

position_test = (
    df.groupby(position_bucket)
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_clicks=("clicks_90d", "mean")
      )
)

display(position_test)


# Signal 3: CTR

ctr_bucket = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_test = (
    df.groupby(ctr_bucket)
      .agg(
          n=("content_id", "count"),
          avg_clicks=("clicks_90d", "mean"),
          avg_position=("avg_position", "mean")
      )
)

display(ctr_test)

/tmp/ipykernel_726/711930662.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(volume_bucket)


,n,avg_impressions,avg_clicks
search_volume,,,
"(-0.001, 10.0]",20860,4950.883941,16.081352
"(10.0, 20.0]",2290,5690.318341,18.963319
"(20.0, 74000.0]",6850,5796.309635,15.187883


/tmp/ipykernel_726/711930662.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(position_bucket)


,n,avg_ctr,avg_clicks
avg_position,,,
"(-0.001, 6.2]",7543,1.079626,33.095718
"(6.2, 10.8]",7534,0.436351,14.308866
"(10.8, 22.3]",7462,0.319024,11.231573
"(22.3, 245.0]",7461,0.202433,5.584506


/tmp/ipykernel_726/711930662.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(ctr_bucket)


,n,avg_clicks,avg_position
ctr,,,
"(-0.001, 0.07]",15224,0.685956,19.218431
"(0.07, 0.29]",7503,16.213248,15.116007
"(0.29, 100.0]",7273,48.237179,11.587323


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:

## Flag-Linked Test

'''I tested the assumption behind the Quick Win flag.

Assumption:
Pages with stronger search demand represent higher-value optimisation opportunities.

Verdict: CONFIRMED

The observed data suggest that pages with higher search volume generally show greater traffic potential. This supports the use of search volume as part of a Quick Win prioritisation rule.'''

'I tested the assumption behind the Quick Win flag.\n\nAssumption:\nPages with stronger search demand represent higher-value optimisation opportunities.\n\nVerdict: CONFIRMED\n\nThe observed data suggest that pages with higher search volume generally show greater traffic potential. This supports the use of search volume as part of a Quick Win prioritisation rule.'

In [ ]:
quickwin_test = (
    df.groupby(
        pd.qcut(
            df["search_volume"].fillna(0),
            q=4,
            duplicates="drop"
        )
    )
    .agg(
        n=("content_id", "count"),
        avg_impressions=("impressions_90d", "mean"),
        avg_clicks=("clicks_90d", "mean"),
        avg_ctr=("ctr", "mean")
    )
)

display(quickwin_test)

/tmp/ipykernel_726/2925169790.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


,n,avg_impressions,avg_clicks,avg_ctr
search_volume,,,,
"(-0.001, 10.0]",20860,4950.883941,16.081352,0.642655
"(10.0, 20.0]",2290,5690.318341,18.963319,0.252389
"(20.0, 74000.0]",6850,5796.309635,15.187883,0.195364


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
## Practical Implications

'''The signal audit suggests that search volume and ranking position are useful indicators when prioritising SEO opportunities.

Content teams should focus first on pages with meaningful search demand that are currently underperforming in visibility or CTR.

These findings should be treated as decision-support evidence rather than proof of causation, but they provide a practical starting point for optimisation efforts.'''

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.